# 📗 텍스트 벡터화와 임베딩 — 군집·토픽·이미지

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

앞선 교안(교안_01)에서 우리는 문장을 **임베딩**(숫자 벡터)으로 바꾸고, **코사인 유사도**로 "두 문장이 얼마나 비슷한가"를 재는 법을 익혔습니다. 이번 교안에서는 그 임베딩을 **실제로 활용**합니다. 정답(라벨) 없이 비슷한 문서끼리 **자동으로 묶는 군집(clustering)**, 각 묶음의 **대표 단어(토픽 기초)**, 그리고 임베딩이 텍스트만이 아니라 **이미지에도** 똑같이 통한다는 것(**CLIP**)까지 확인합니다.

## ⏪ 복습 — 지난 시간(교안_01): 임베딩·코사인 유사도
- **임베딩**: 문장을 의미가 담긴 **고정 길이 숫자 벡터**로 바꾸는 것. 뜻이 비슷한 문장은 벡터도 가까워집니다.
- **모델**: `SentenceTransformer('jhgan/ko-sroberta-multitask')` 로 한국어 문장을 **768차원** 벡터로 인코딩했습니다.
- **코사인 유사도**: 두 벡터가 이루는 각도로 유사도를 재는 방법(−1~1, 1에 가까울수록 비슷). "검색"·"추천"의 기초가 됩니다. 유클리드 거리·내적도 함께 재 봤고, **L2 정규화**하면 셋의 순위가 일치한다는 것까지 확인했습니다.
- **UMAP**: 768차원 임베딩을 **2차원 좌표**로 줄여 그림으로 확인했습니다 — 오늘 군집도 이 좌표 위에서 합니다.

> 오늘은 이 임베딩 위에 **비지도 학습(군집)** 을 얹습니다. 라벨을 주지 않아도 임베딩만으로 비슷한 리뷰가 스스로 모이는지, 그 결과가 실제 장르와 얼마나 맞는지 눈으로 확인합니다.

**오늘의 목표**
- [ ] **KMeans 군집**으로 정답 없이 비슷한 문서를 묶고, 결과를 실제 라벨과 대조해 본다.
- [ ] **엘보우 방법**으로 "군집을 몇 개로 나눌지(k)"를 관성(inertia)으로 가늠한다.
- [ ] **실루엣 계수**로 군집이 얼마나 잘 뭉쳤는지 정량 평가하고 여러 k를 비교한다.
- [ ] 각 군집의 **대표 키워드**를 TF-IDF 로 뽑아 **기초 토픽 모델링**을 경험한다.
- [ ] **CLIP** 으로 **이미지도 임베딩**해 같은 카테고리끼리 모이는지 확인하고, **텍스트-이미지 멀티모달** 검색을 해 본다.

In [ ]:
# 오늘 실습에 쓸 라이브러리와 한글 폰트를 준비합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.feature_extraction.text import TfidfVectorizer
from umap import UMAP
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

---
## 데이터 준비 — 영화 리뷰 임베딩 다시 만들기

교안_01과 **같은 데이터**(장르별 영화 리뷰 15개)로 이어집니다. 군집을 하려면 먼저 각 리뷰를 **임베딩 벡터**로 바꿔야 하므로, 지난 시간의 인코딩을 한 셀에 재현합니다. 새 데이터를 불러오면 언제나 **먼저 살펴보는 것**이 습관입니다(`head` 로 앞부분 확인).

In [ ]:
movie_df = pd.read_csv('data/movie_reviews.csv')
print('리뷰 수:', len(movie_df))
print('장르 분포:'); print(movie_df['genre'].value_counts())
display(movie_df.head())

In [ ]:
# 한국어 문장 임베딩 모델(교안_01과 동일)로 15개 리뷰를 벡터로 변환
text_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
movie_emb = text_model.encode(movie_df['review'].tolist())
print('임베딩 배열 shape =', movie_emb.shape, '  (리뷰 15개 × 768차원)')

## 군집 전에 — 임베딩을 2차원으로 줄입니다 (UMAP)

임베딩은 **768차원**입니다. 그런데 차원이 높으면 뒤에서 배울 **거리 기반 지표(실루엣 계수)가 잘 안 통하고**, 짧은 문장이 몇 개뿐일 때는 노이즈가 커서 군집 경계도 흐릿해집니다. 그래서 교안_01에서 배운 **UMAP** 으로 임베딩을 **2차원 좌표로 요약**한 뒤, 그 위에서 **군집·평가·시각화를 한 공간에서** 합니다.

- 만든 **`reducer` 객체를 남겨 둡니다**: 나중에 **새 문장**도 같은 UMAP 으로 변환(`reducer.transform`)해야 같은 좌표계에서 군집을 예측할 수 있기 때문입니다.

> 참고: 앞 교안에서 문장 유사도는 **코사인**으로 쟀는데, KMeans·실루엣은 이 좌표에서 **거리(유클리드)** 로 가까움을 봅니다. UMAP 은 **가까운 이웃 관계를 지키도록** 점을 다시 배치하므로, 원래 공간에서 가깝던 점들이 좌표에서도 대체로 가깝게 놓입니다.

In [ ]:
# UMAP으로 768차원 임베딩을 2차원 좌표로 요약 (군집·평가·시각화를 이 좌표에서 진행)
reducer = UMAP(n_components=2, n_neighbors=5, min_dist=0.05, random_state=0)
movie_coords = reducer.fit_transform(movie_emb)
print('2차원 좌표 shape =', movie_coords.shape, '  (리뷰 15개 × 2차원)')

> ⚠️ **눌렀으니 잃은 것도 있습니다.** UMAP 은 이웃 관계를 지키느라 덩어리를 **실제보다 또렷하게** 그립니다. 뒤에서 볼 **실루엣 0.93 은 이 2차원 좌표에서 잰 값**이라는 점을 꼭 기억하세요 — **768차원 원본에서 그대로 재면 0.17 수준**입니다. 즉 "0.93 이니 군집이 완벽하다"가 아니라 "2차원으로 요약한 그림 위에서는 0.93"이 정확한 말입니다.

> 또 2차원 그림에서 **가까워 보이는 두 점이 같은 군집이 아닐 수도** 있습니다. 그림은 768차원을 눌러 만든 **요약**이지 원본이 아니니까요.

> **실무 기본형은 임베딩 공간에서 바로 군집하고, 2차원은 눈으로 볼 때만** 씁니다. 여기서는 엘보우·실루엣 같은 지표를 **눈에 보이는 그림과 같은 공간**에서 다루려고 예외적으로 좌표 위에서 진행합니다. 다음 단원에서는 임베딩 공간에서 군집합니다.

---
# 1. KMeans 군집 — 정답 없이 비슷한 것끼리 묶기

## 왜 필요할까요?
지금까지의 분석은 대부분 **정답(라벨)** 이 있었습니다. "이 리뷰는 액션"처럼요. 그런데 현실에서는 **라벨이 없는** 데이터가 훨씬 많습니다. 수천 개의 고객 문의를 누가 일일이 분류할까요? 이때 "정답을 주지 않아도 **비슷한 것끼리 스스로 묶게** 하는" 방법이 **군집(clustering)** 이고, 이렇게 정답 없이 배우는 방식을 **비지도 학습(unsupervised learning)** 이라고 합니다.

## KMeans 의 아이디어 — 중심점을 옮겨 가며 묶는다
**KMeans** 는 가장 널리 쓰이는 군집 알고리즘입니다. 이름의 K는 "몇 개의 묶음(군집)으로 나눌지"이고, means(평균)는 각 군집의 **중심점(centroid)** 을 뜻합니다. 동작은 아주 단순한 **두 단계의 반복**입니다.

1. **할당**: 각 점(임베딩 벡터)을 **가장 가까운 중심점**의 군집에 배정한다.
2. **갱신**: 각 군집에 모인 점들의 **평균 위치**로 중심점을 옮긴다.

이 두 단계를 중심점이 더 이상 움직이지 않을 때까지 반복하면, 가까운 점끼리 자연스럽게 한 군집이 됩니다. 우리는 각 리뷰를 임베딩한 뒤 **UMAP으로 2차원 좌표**(`movie_coords`)로 요약해 뒀으니, "뜻이 비슷한 리뷰 = 좌표가 가까운 점"이 되어 **의미가 비슷한 리뷰끼리 묶이기**를 기대할 수 있습니다.


<img src="images/KMeans_반복.png" width="900">

> `KMeans(n_clusters=3, random_state=0, n_init=10)` — 3개로 나누고, 초기 중심점 위치에 따라 결과가 달라질 수 있어 `n_init=10`(10번 시도 중 best)·`random_state=0`(재현성)을 함께 줍니다.

In [ ]:
# 2차원 좌표를 3개 군집으로 나눈다 (정답 라벨은 주지 않는다)
km = KMeans(n_clusters=3, random_state=0, n_init=10)
cluster_labels = km.fit_predict(movie_coords)

result = movie_df.copy()
result['cluster'] = cluster_labels
display(result[['cluster', 'genre', 'review']])

In [ ]:
# 군집 결과가 '실제 장르'와 얼마나 맞는지 교차표로 대조
# (군집 번호 자체엔 의미가 없다 — 어느 장르와 겹치는지가 핵심)
compare = pd.crosstab(result['cluster'], result['genre'])
print('[군집 × 실제 장르 교차표]')
display(compare)

## 해석 — 라벨 없이도 장르가 갈렸다

교차표를 보면 **각 군집이 정확히 한 장르에만 대응**합니다(한 칸에 5씩, 나머지는 0인 **대각선 모양**). 즉 액션·로맨스·코미디 리뷰가 **정답을 주지 않았는데도** 저마다 다른 군집으로 깔끔히 갈렸습니다. 임베딩이 리뷰의 **의미**를 잘 담고 있어서, 뜻이 비슷한 문장끼리 벡터 공간에서 가까이 모였기 때문입니다.

- **군집 번호는 임의**입니다(0·1·2 라는 숫자에 순서·의미 없음). 어떤 실제 장르와 겹치는지는 교차표로 읽습니다.
- 이것이 비지도 학습의 힘입니다 — 라벨을 만드는 비용 없이 **데이터 스스로의 구조**를 드러냅니다.

> ⚠️ **이 데이터가 유난히 쉽다는 점은 알고 넘어갑시다.** 리뷰 15건이 장르 어휘를 아주 또렷하게 담고 있어 교차표가 완벽한 대각선으로 나왔습니다. **실무 데이터는 이렇지 않습니다** — 로맨틱 코미디처럼 경계에 걸친 문서, 장르를 짐작할 단서가 없는 짧은 글이 섞여 있어 군집이 지저분하게 갈립니다. 그래서 다음 절들에서 **얼마나 잘 갈렸는지 재는 법**(엘보우·실루엣)을 배웁니다.

아래는 임베딩을 2차원으로 줄여(UMAP) 군집을 그린 그림입니다. 세 덩어리가 눈으로도 분명히 갈립니다. **직접 그려 봅시다** — 교안_01 7절의 산점도와 같은 방식이고, 색만 장르 대신 **군집 번호**로 줍니다.

<img src="images/군집_산점도.png" width="640"/>

In [ ]:
# 군집 결과를 2차원 좌표 위에 색으로 그린다 (실제 장르는 글씨로 달아 대조)
colors = ['#2563eb', '#f59e0b', '#10b981']
plt.figure(figsize=(7, 5.5))
for i in range(3):
    m = cluster_labels == i
    plt.scatter(movie_coords[m, 0], movie_coords[m, 1],
                color=colors[i], s=80, label=f'군집 {i}', alpha=0.8)
for i in range(len(movie_df)):
    plt.annotate(movie_df['genre'][i], (movie_coords[i, 0], movie_coords[i, 1]),
                 fontsize=8, alpha=0.7)
plt.title('영화 리뷰 임베딩 KMeans 군집 (UMAP 2차원)')
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2'); plt.legend(); plt.tight_layout(); plt.show()
print('같은 색(군집)끼리 모였고, 글씨로 단 실제 장르도 색과 일치한다')

### 🖐️ 함께 따라하기 — 새 리뷰는 어느 군집에 들어갈까
학습된 KMeans 모델(`km`)로 **처음 보는 리뷰**의 군집을 예측해 보세요. 새 문장을 `text_model.encode(...)` 로 임베딩하고 `reducer.transform(...)` 으로 **같은 2차원 좌표계**로 바꾼 뒤 `km.predict(...)` 에 넣습니다(군집은 2차원 좌표에서 학습했으니 새 점도 같은 좌표로 맞춰야 해요). 예측한 군집 번호를 위 교차표와 맞춰 보면 어느 장르로 분류됐는지 알 수 있습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 새 리뷰 문장 2개를 리스트 new_reviews 로 만든다 (장르가 다른 문장으로)
# 2) text_model.encode(new_reviews) 로 임베딩한 뒤 reducer.transform(...) 으로 좌표 new_coords 를 만든다
# 3) km.predict(new_coords) 로 각 문장의 군집 번호 new_pred 를 예측한다
# 4) zip 으로 문장과 예측 군집을 짝지어 출력한다

### ✅ 바로 확인 퀴즈
**1.** KMeans 처럼 정답(라벨) 없이 데이터의 구조를 배우는 방식을 무엇이라고 하나요?

<details><summary>정답 보기</summary>

**비지도 학습(unsupervised learning)** 입니다. 정답 라벨을 주고 배우는 지도 학습과 달리, 군집은 라벨 없이 "비슷한 것끼리" 스스로 묶습니다.

</details>

**2.** KMeans 는 어떤 두 단계를 반복하나요?

<details><summary>정답 보기</summary>

① **할당**: 각 점을 가장 가까운 **중심점**의 군집에 배정 → ② **갱신**: 각 군집 점들의 **평균**으로 중심점을 이동. 중심점이 더 이상 움직이지 않을 때까지 반복합니다.

</details>

**3.** 군집 결과의 "군집 0, 1, 2" 번호는 그 자체로 순서나 의미가 있나요?

<details><summary>정답 보기</summary>

**없습니다.** 번호는 임의로 붙은 이름표일 뿐입니다. 어떤 군집이 무엇을 뜻하는지는 각 군집에 모인 데이터(또는 실제 라벨과의 교차표·대표 키워드)로 해석합니다.

</details>

---
## 1-2. 거리로 묶는다는 말의 함정 — 축마다 단위가 다르면

### 왜 이 이야기를 지금 하나요?
KMeans 는 **거리**로 묶습니다. 그런데 거리는 **모든 축을 똑같이 취급**합니다 — 한 축에서 1만큼 떨어진 것과 다른 축에서 1만큼 떨어진 것을 **같은 크기**로 봅니다.

방금 우리가 군집한 것은 **임베딩을 UMAP으로 줄인 좌표**였습니다. 두 축 모두 같은 성질·비슷한 범위라 문제가 없었죠. 그런데 **현실의 표 데이터**는 그렇지 않습니다. 무게는 그램, 길이는 센티미터처럼 **단위가 뒤섞여** 있습니다. 이때 무슨 일이 벌어지는지 직접 봅시다.

### 생선 데이터로 확인하기
생선 159마리의 **무게·길이·대각선길이·높이·너비** 5개 측정값이 있습니다(*혼자 공부하는 머신러닝+딥러닝*, 박해선). 종(Species)은 **정답 라벨**이라 군집에는 쓰지 않고, **나중에 채점용**으로만 씁니다.

In [ ]:
import pandas as pd
fish = pd.read_csv('data/fish.csv')
print('행·열:', fish.shape, '| 종:', fish['Species'].nunique(), '가지')
display(fish.head(3))

features = ['Weight', 'Length', 'Diagonal', 'Height', 'Width']
print('\n[각 특성의 범위 — 단위가 뒤섞여 있다]')
display(fish[features].describe().loc[['min', 'max', 'std']].round(1))

### 무게 하나가 거리를 독차지합니다

무게는 0~1650(g), 너비는 1.0~8.1 입니다. 유클리드 거리는 **각 축의 차이를 제곱해 더하므로**, 숫자가 큰 축이 그대로 지배합니다. 얼마나 지배하는지 분산으로 재 봅시다.

In [ ]:
X = fish[features].values
var = X.var(axis=0)
for name, v in zip(features, var):
    print(f'{name:9} 분산 {v:10.1f}   거리에 기여하는 비중 {v / var.sum() * 100:5.2f}%')
print('\n→ 무게 하나가 사실상 전부다. 나머지 네 특성은 있으나 마나다.')

### 표준화(StandardScaler) — 모든 축을 같은 자로 재기

**표준화**는 각 특성을 **평균 0, 표준편차 1** 로 바꿉니다. 단위를 없애고 "평균에서 표준편차 몇 개만큼 떨어졌나"라는 **공통의 자**로 바꾸는 것입니다.

$$ z = \frac{x - \mu}{\sigma} $$

### 문법
- **`StandardScaler().fit_transform(X)`** → 표준화된 배열. `fit`(평균·표준편차 학습) + `transform`(변환)을 한 번에.

> 아래에서 **표준화 전후로 같은 KMeans** 를 돌려, 실제 생선 종과 얼마나 맞는지 **ARI** 로 비교합니다. (ARI는 군집이 정답 라벨과 얼마나 일치하는지를 재는 지표로, 1에 가까울수록 잘 맞은 것입니다.)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

X_scaled = StandardScaler().fit_transform(X)
print('표준화 후 평균:', X_scaled.mean(axis=0).round(2), '/ 표준편차:', X_scaled.std(axis=0).round(2))

for tag, data in [('원본 그대로', X), ('표준화 후  ', X_scaled)]:
    labels = KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict(data)
    ari = adjusted_rand_score(fish['Species'], labels)
    sil = silhouette_score(data, labels)
    print(f'{tag} : 실제 종과의 일치도(ARI) {ari:.3f}   실루엣 {sil:.3f}')

### 읽는 법 — 두 숫자가 서로 다른 말을 합니다

- **ARI 는 0.134 → 0.302 로 2배 넘게 올랐습니다.** 표준화하고 나서야 무게 말고 **생김새(길이·높이·너비)** 까지 반영되어 실제 종에 가깝게 묶였습니다.
- 그런데 **실루엣은 오히려 0.608 → 0.456 으로 떨어졌습니다.** 실루엣은 "기하학적으로 잘 뭉쳤나"만 보는데, 원본은 무게 한 축만 보고 나누니 **모양은 깔끔**했던 겁니다. 정작 생선 종과는 안 맞으면서요.

> **지표 하나만 보면 속습니다.** 실루엣이 높다고 좋은 군집이 아닙니다. 정답 라벨이 있다면 ARI 처럼 **정답과 맞춰 보는 지표**를 함께 봐야 합니다.

### 그럼 임베딩에도 표준화를 해야 하나요?
**아닙니다.** 임베딩의 768개 칸은 **모두 같은 성질·같은 스케일**입니다 — 어느 칸도 "그램"이거나 "센티미터"가 아닙니다. 여기에 표준화를 걸면 오히려 **의미 있는 축의 강약을 뭉개** 버립니다.

| 데이터 | 표준화 | 왜 |
|---|---|---|
| **표 데이터**(무게·길이·가격…) | **필요** | 단위가 뒤섞여 큰 숫자 축이 거리를 독차지 |
| **임베딩**(768차원) | **불필요** | 모든 칸이 같은 성질·같은 스케일 |

판단 기준은 하나입니다 — **"축마다 단위가 다른가?"**

### ✅ 바로 확인 퀴즈

**1.** 무게(0~1650g)와 너비(1.0~8.1)를 그대로 두고 KMeans 를 돌리면 무슨 일이 생기나요?

<details><summary>정답 보기</summary>

숫자가 큰 **무게가 거리를 독차지**해, 사실상 무게 하나로만 묶입니다(실측: 무게가 전체 분산의 **99.79%**). 나머지 네 특성은 결과에 거의 영향을 주지 못합니다.

</details>

**2.** 표준화 후 실루엣이 **더 낮아졌는데도** 표준화가 낫다고 보는 이유는?

<details><summary>정답 보기</summary>

실루엣은 **정답을 모른 채 모양만** 보는 지표라, 한 축으로만 자르면 오히려 깔끔해 보입니다. 정답(생선 종)과 맞춰 보는 **ARI 는 0.134 → 0.302 로 크게 올랐습니다.** **목적에 맞는 지표**로 판단해야 합니다.

</details>

---
# 2. 군집을 몇 개로 나눌까 — 엘보우 방법

## 왜 필요할까요?
방금은 `n_clusters=3` 을 우리가 정해 줬습니다. 하지만 실제로는 **몇 개로 나눠야 좋은지 미리 알 수 없습니다**. 이때 도움을 주는 것이 **엘보우 방법(elbow method)** 입니다.

## 관성(inertia)과 팔꿈치
KMeans 는 **관성(inertia)** 이라는 값을 남깁니다. 관성은 "각 점이 자기 군집 중심점에서 얼마나 멀리 있나"의 **제곱 거리 합**입니다. 즉 **작을수록 군집이 촘촘히 뭉친** 것입니다. 군집 수 k 를 늘리면 관성은 **항상 줄어듭니다** (극단적으로 점마다 군집이면 0). 그래서 "관성이 작다"만으로 좋은 k 를 고를 수는 없습니다.

대신 **k 를 늘려도 관성이 더 이상 크게 줄지 않는 지점** — 그래프가 팔을 굽힌 **팔꿈치(elbow)** 모양이 되는 k 를 고릅니다. 그 전까지는 나눌수록 확 촘촘해지다가, 그 뒤로는 나눠 봐야 별 이득이 없다는 뜻입니다.

In [ ]:
# k = 1..8 각각에 대해 관성(inertia)을 계산
k_range = range(1, 9)
inertias = []
for k in k_range:
    km_k = KMeans(n_clusters=k, random_state=0, n_init=10).fit(movie_coords)
    inertias.append(km_k.inertia_)
    print(f'k={k}  관성(inertia) = {km_k.inertia_:.1f}')

# 꺾은선으로 그려 '팔꿈치'를 눈으로 찾는다
plt.figure(figsize=(7, 4.5))
plt.plot(list(k_range), inertias, 'o-', color='#2563eb')
plt.axvline(3, color='red', ls='--', alpha=0.6, label='팔꿈치 후보 k=3')
plt.title('엘보우 방법 — k별 관성(inertia)')
plt.xlabel('군집 수 k'); plt.ylabel('관성 (inertia)')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## 해석 — 팔꿈치는 k=3 근처

관성은 k 가 커질수록 계속 줄지만, **줄어드는 폭**은 점점 작아집니다. k=1→2→3 까지는 관성이 크게 떨어지다가 **k=3 을 지나면서 곡선이 완만**해집니다. 즉 **팔꿈치가 k=3 근처**입니다 — 실제 장르가 3개(액션·로맨스·코미디)인 것과도 잘 맞습니다.

> 엘보우는 **칼같은 정답이 아니라 안내선**입니다. 팔꿈치가 애매할 때가 많아, 다음 절의 **실루엣 계수** 같은 정량 지표와 **함께** 보고 결정합니다.

<img src="images/엘보우.png" width="600"/>

### 🖐️ 함께 따라하기 — 관성 감소량으로 팔꿈치 찾기
그림 대신 **숫자로** 팔꿈치를 확인해 보세요. 이웃한 k 사이의 **관성 감소량**(앞 k − 뒤 k)을 출력하면, 감소량이 확 작아지는 지점이 팔꿈치입니다. 위에서 만든 `inertias` 리스트를 그대로 사용하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) for i in range(1, len(inertias)): 로 이웃한 k 를 훑는다
# 2) drop = inertias[i-1] - inertias[i] 로 관성 감소량을 구한다
# 3) f-string 으로 'k=i -> k=i+1  감소량 ...' 형태로 출력한다
# 4) 감소량이 확 작아지기 시작하는 k 가 팔꿈치임을 확인한다

### ✅ 바로 확인 퀴즈
**1.** 군집 수 k 를 늘리면 관성(inertia)은 어떻게 되나요?

<details><summary>정답 보기</summary>

**항상 줄어듭니다.** 군집이 많아질수록 각 점이 더 가까운 중심점을 갖기 때문입니다. 그래서 "관성이 작다"만으로는 좋은 k 를 고를 수 없고, **감소가 완만해지는 팔꿈치**를 봅니다.

</details>

**2.** 엘보우 그래프에서 우리가 고르는 지점은 어디인가요?

<details><summary>정답 보기</summary>

**팔꿈치(elbow)** — k 를 늘려도 관성이 더는 크게 줄지 않기 시작하는 지점입니다. 그 전까지는 나눌수록 촘촘해지지만, 그 뒤로는 나눠도 이득이 적다는 뜻입니다.

</details>

---
# 3. 군집 품질 — 실루엣 계수

## 왜 필요할까요?
엘보우는 팔꿈치가 애매할 때가 많습니다. **군집이 얼마나 잘 뭉쳤는지**를 하나의 숫자로 재는 지표가 **실루엣 계수(silhouette coefficient)** 입니다. 여러 k 를 실루엣으로 비교하면 더 객관적으로 고를 수 있습니다.

## 실루엣 계수의 직관 — a 는 작게, b 는 크게
각 점 하나에 대해 두 거리를 봅니다.

- **a** = 그 점과 **같은 군집** 안 다른 점들까지의 **평균 거리** (작을수록 자기 무리에 잘 붙음)
- **b** = 그 점에서 **가장 가까운 다른 군집** 점들까지의 **평균 거리** (클수록 옆 무리와 잘 떨어짐)

실루엣 계수는 $s = (b - a) / \max(a, b)$ 로, **−1 ~ 1** 값을 갖습니다.

- **1 에 가까움**: 자기 군집엔 바짝(a 작음), 옆 군집과는 멀리(b 큼) — 잘 뭉친 점.
- **0 근처**: 두 군집 경계에 걸친 점.
- **음수**: 옆 군집이 더 가까움 — 잘못 묶였을 수 있는 점.

전체 점의 평균이 그 군집화의 **실루엣 점수**입니다. `silhouette_score`(평균)와 `silhouette_samples`(점별 값)를 사용합니다.

In [ ]:
# 여러 k 에 대해 실루엣 점수(평균)를 비교
print('k별 실루엣 계수(평균):')
for k in range(2, 7):
    labels_k = KMeans(n_clusters=k, random_state=0, n_init=10).fit_predict(movie_coords)
    score = silhouette_score(movie_coords, labels_k)
    print(f'  k={k}  실루엣 계수 = {score:.4f}')

In [ ]:
# k=3 의 점별 실루엣 값으로 '실루엣 플롯'을 그린다
sample_sil = silhouette_samples(movie_coords, cluster_labels)
avg_sil = silhouette_score(movie_coords, cluster_labels)
colors = ['#2563eb', '#f59e0b', '#10b981']

plt.figure(figsize=(7, 5))
y_lower = 0
for i in range(3):
    vals = np.sort(sample_sil[cluster_labels == i])
    y_upper = y_lower + len(vals)
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, color=colors[i], alpha=0.8)
    plt.text(-0.02, y_lower + len(vals) / 2, f'군집 {i}')
    y_lower = y_upper
plt.axvline(avg_sil, color='red', ls='--', label=f'평균 {avg_sil:.3f}')
plt.title('실루엣 플롯 (k=3)')
plt.xlabel('실루엣 계수'); plt.ylabel('문서')
plt.legend(); plt.tight_layout(); plt.show()

## 해석 — k=3 이 가장 낫고, 엘보우와 일치

- 여러 k 중 **k=3 의 실루엣 점수가 가장 높습니다**(약 0.93). 앞 절의 엘보우가 가리킨 k=3 과 **일치**하고, 실제 장르 수(3개)와도 맞습니다.
- 다만 **0.93 이라는 크기 자체를 믿지는 마세요.** 이 값은 **UMAP 이 이웃 관계를 살려 또렷하게 펼쳐 놓은 2차원 좌표에서 잰 값**입니다. 같은 군집을 **768차원 원본에서 재면 0.17** 수준으로 뚝 떨어집니다. 즉 UMAP 이 군집을 '더 잘' 만든 게 아니라 **더 잘 보이게 그린 것**입니다. 우리가 여기서 쓰는 정보는 '0.93 이라는 크기'가 아니라 **'여러 k 중 k=3 이 가장 높다'는 순위**입니다.
- 실루엣 플롯에서 **음수 막대가 거의 없고** 세 군집의 폭이 고른 것도 좋은 신호입니다.

> 정리: **엘보우 + 실루엣 + (있다면) 실제 라벨** 을 함께 보고 k 를 정합니다. 여기서는 셋이 모두 k=3 을 가리켰습니다.

<img src="images/실루엣.png" width="600"/>

### 🖐️ 함께 따라하기 — 군집별 평균 실루엣 비교
위에서 만든 점별 실루엣 값(`sample_sil`)으로, **각 군집의 평균 실루엣**을 구해 보세요. 어떤 군집이 가장 잘 뭉쳤는지(평균이 큰지) 비교할 수 있습니다. `cluster_labels` 로 군집별 값을 골라 평균을 냅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) for i in sorted(set(cluster_labels)): 로 군집 번호를 훑는다
# 2) sample_sil[cluster_labels == i] 로 그 군집에 속한 점들의 실루엣 값만 고른다
# 3) .mean() 으로 평균을 내어 f-string 으로 출력한다

### ✅ 바로 확인 퀴즈
**1.** 실루엣 계수에서 a 와 b 는 각각 무엇인가요?

<details><summary>정답 보기</summary>

**a** = 같은 군집 안 다른 점들까지의 평균 거리(작을수록 좋음), **b** = 가장 가까운 **다른** 군집까지의 평균 거리(클수록 좋음)입니다. $s=(b-a)/\max(a,b)$ 로 계산합니다.

</details>

**2.** 실루엣 계수가 **음수**인 점은 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

자기 군집보다 **옆 군집이 더 가깝다**는 뜻으로, **잘못 묶였을 가능성**이 있는 점입니다. 음수 점이 많으면 그 군집화(또는 k)가 데이터에 잘 맞지 않는 신호입니다.

</details>

**3.** 좋은 k 를 정할 때 실루엣 점수를 어떻게 쓰나요?

<details><summary>정답 보기</summary>

여러 k 에 대해 실루엣 점수를 구해 **가장 높은 k** 를 후보로 봅니다. 엘보우·실제 라벨(있으면)과 **함께** 보고 결정합니다. 여기서는 k=3 이 실루엣 최고이자 엘보우·장르 수와도 일치했습니다.

</details>

---
# 4. 클러스터 대표 키워드 — 토픽 모델링의 기초

## 왜 필요할까요?
군집을 나눴지만 "군집 0, 1, 2"라는 번호만으로는 **무슨 주제의 묶음인지** 알 수 없습니다. 각 군집의 문서를 모아 **그 묶음을 대표하는 단어**를 뽑으면, "아, 이 군집은 액션 이야기구나"처럼 **주제에 이름표**를 달 수 있습니다. 이렇게 문서 묶음에서 주제를 요약하는 일이 **토픽 모델링(topic modeling)** 이고, 여기서는 그 **가장 기본형**을 TF-IDF 로 해 봅니다.

## TF-IDF 로 대표 단어 뽑기
**TF-IDF** 는 "이 묶음에서 자주 나오면서(TF), 다른 묶음에는 드문(IDF)" 단어에 높은 점수를 주는 방법입니다. 그래서 세 군집에 두루 흔한 말보다 **그 군집을 특징짓는 단어**가 위로 올라오기를 기대합니다. 절차는 단순합니다.

1. 각 군집에 속한 리뷰들을 **하나의 문서로 합친다**(군집당 문서 1개).
2. `TfidfVectorizer` 로 세 문서를 벡터화한다.
3. 각 군집 문서에서 **TF-IDF 점수가 높은 상위 단어**를 뽑는다.

> 이것은 토픽 모델링의 **입문형**입니다. 아래에서 결과를 보고 **어디까지 되고 어디서부터 안 되는지**를 함께 확인합니다 — 그 한계가 다음 단원(임베딩 기반 토픽 모델링)이 필요한 이유입니다.

In [ ]:
# 1) 군집별로 리뷰를 하나의 문서로 합친다
cluster_docs = [' '.join(movie_df['review'][cluster_labels == i]) for i in range(3)]

# 2) TF-IDF 벡터화 (한 글자 단어도 살리려 token_pattern 지정)
tfidf = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
tfidf_matrix = tfidf.fit_transform(cluster_docs)
feature_names = tfidf.get_feature_names_out()

# 3) 각 군집에서 TF-IDF 점수 상위 6개 단어를 뽑는다
print('군집별 대표 단어(TF-IDF 상위 6개):')
for i in range(3):
    scores = tfidf_matrix[i].toarray().ravel()
    top_idx = scores.argsort()[::-1][:6]
    keywords = ', '.join(feature_names[top_idx])
    print(f'  군집 {i}: {keywords}')

### 점수도 함께 봅시다 — 왜 "상위 6개"가 애매한가
단어만 보면 그럴듯하지만, **점수를 같이 찍어 보면** 이야기가 달라집니다. IDF 가 몇 종류나 되는지, 상위 단어들의 점수가 서로 다른지 확인해 보세요.

In [ ]:
# 대표 단어의 '점수'까지 같이 본다
scores0 = tfidf_matrix[0].toarray().ravel()
top0 = scores0.argsort()[::-1][:6]
print('군집 0 상위 6개와 점수:')
for j in top0:
    print(f'   {feature_names[j]:12} {scores0[j]:.4f}')

# IDF 가 단어를 얼마나 구별하고 있나
n_unique_idf = len(np.unique(np.round(tfidf.idf_, 4)))   # 서로 다른 IDF 값의 개수
n_words = len(feature_names)
print(f'\n서로 다른 IDF 값의 개수: {n_unique_idf} / 전체 단어 수: {n_words}')

## 해석 — 번호에 주제 이름이 붙는다

**절반은 성공, 절반은 한계**입니다. 액션 군집에서 `히어로가·악당·싸워요`, 코미디 군집에서 `오해가·웃음으로·엉뚱한` 이 올라온 것은 기대한 대로입니다. 그런데 로맨스 군집의 1위는 **`두`** 라는 조사고, `연인의`·`따뜻한` 은 상위 6개에 들지 못했습니다.

### 왜 이렇게 됐나 — 점수를 보면 답이 나옵니다
위에서 점수를 찍어 보면 상위 단어들의 TF-IDF 가 **거의 다 같은 값**이고, **서로 다른 IDF 값이 단 2개**뿐입니다. 이유는 둘입니다.

1. **문서가 3개뿐** — IDF 는 "몇 개의 문서에 나오는가"로 계산되는데, 군집당 문서 1개씩 총 3개라 거의 모든 단어가 "1개 문서에만 등장"으로 똑같이 취급됩니다. **IDF 가 단어를 구별하지 못합니다.**
2. **띄어쓰기 단위로 세기 때문** — `연인의`·`연인으로` 가 서로 다른 단어로 흩어져 각각 1회가 됩니다. "연인"이라는 뜻은 두 번 나왔는데 점수는 반씩 나뉩니다.

> 점수가 동률이면 순서는 사실상 **먼저 나온 단어 순**으로 정해집니다. 즉 지금 보이는 "상위 6개"는 TF-IDF 가 골라 준 것이라기보다 **동점자 명단**에 가깝습니다. 대표 키워드를 뽑을 때는 **단어만 보지 말고 점수도 함께 보는 습관**이 필요합니다.

### 그럼 어떻게 개선하나
- **문서를 더 잘게** — 군집당 1개로 합치지 말고 리뷰 하나하나를 문서로 두면 IDF 가 살아납니다 (다만 이 데이터는 15건이 서로 겹치는 단어가 거의 없어 큰 효과는 없습니다 — 데이터가 작은 탓입니다).
- **형태소 분석·불용어 제거**(11일차) — `연인의`·`연인으로` 를 `연인` 으로 묶고 조사 `두` 를 걸러냅니다.
- **단어 세기를 벗어나기** — 애초에 단어 빈도가 아니라 **임베딩**으로 주제를 뽑는 방법이 있습니다. 그것이 **다음 단원**에서 배울 본격 토픽 모델링이고, 오늘 이 한계가 바로 그 필요성입니다.

### 🖐️ 함께 따라하기 — 동점자가 몇 명인지 세어 보기
시연에서는 군집 0 만 점수를 봤습니다. 이번엔 **세 군집 모두** 상위 6개의 점수를 표로 만들어, "몇 개나 같은 점수인지"를 직접 세어 보세요. 대표 키워드를 믿어도 될지 판단하는 습관입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 빈 리스트 rows 를 만들고 for i in range(3): 으로 세 군집을 훑는다
# 2) 군집 i 의 점수 배열에서 상위 6개 인덱스(top_idx)와 그 점수(round(4))를 구한다
# 3) 군집 번호·상위 6개 단어·점수 리스트·set() 으로 센 '서로 다른 점수 수' 를 딕셔너리로 rows 에 담는다
# 4) pd.DataFrame(rows) 로 표를 만들어 display 하고, 동률이 얼마나 많은지 확인한다

### ✅ 바로 확인 퀴즈
**1.** TF-IDF 가 높은 점수를 주는 단어는 어떤 단어인가요?

<details><summary>정답 보기</summary>

**그 묶음에서는 자주(TF) 나오지만 다른 묶음에는 드문(IDF)** 단어입니다. 그래서 모든 군집에 흔한 말보다 **그 군집을 특징짓는 단어**가 위로 올라옵니다.

</details>

**2.** 군집의 대표 단어를 뽑는 일은 무엇의 기초라고 했나요?

<details><summary>정답 보기</summary>

**토픽 모델링(topic modeling)** 의 기초입니다. 문서 묶음에서 주제를 요약해 이름을 붙이는 것으로, 임베딩을 쓴 본격 토픽 모델링은 다음 단원에서 배웁니다.

</details>

---
# 5. 이미지 임베딩 — CLIP

## 왜 필요할까요?
임베딩은 텍스트만의 것이 아닙니다. **이미지도 벡터로** 바꿀 수 있고, 원리는 똑같습니다 — "비슷한 이미지는 벡터도 가깝다". 그러면 앞에서 배운 **코사인 유사도·군집·시각화**를 이미지에도 그대로 쓸 수 있습니다.

## CLIP — 이미지와 텍스트를 같은 공간에 담는 모델
**CLIP**(Contrastive Language–Image Pre-training)은 이미지와 텍스트를 **하나의 공유 벡터 공간**에 임베딩하도록 학습된 모델입니다. 우리는 `SentenceTransformer('clip-ViT-B-32')` 로 사진을 512차원 벡터로 바꿉니다. `data/photos/` 에는 **고양이·자동차·배** 사진이 카테고리별로 5장씩(총 15장) 있습니다. `PIL.Image.open` 으로 열어 `encode` 에 넘기면 됩니다.

> **먼저 알아둘 것 — 코사인의 '기준선'은 모델마다 다릅니다.** 교안_01 에서 문장 임베딩의 코사인은 무관하면 0 근처였습니다. 그런데 CLIP 은 **아무 상관 없는 사진끼리도 0.5 안팎**(높으면 0.7 이상)이 나오고, **글과 사진 사이는 서로 맞아도 0.25 안팎**으로 훨씬 낮게 나옵니다. 모델마다 벡터가 퍼져 있는 방식이 달라서입니다. 그러니 **절대값이 크다/작다로 판단하지 말고, 같은 모델 안에서 어느 쪽이 더 큰지(상대 비교)** 를 보세요. 아래에서도 "0.71 vs 0.51" 처럼 **차이**를 읽습니다.

In [ ]:
# CLIP 모델 로드 (이미지·텍스트 공용 임베딩)
clip_model = SentenceTransformer('clip-ViT-B-32')

# photos/ 의 15장을 카테고리 순서대로 불러온다
categories = ['cat', 'car', 'ship']
image_paths, image_cats = [], []
for cat in categories:
    for i in range(1, 6):
        image_paths.append(f'data/photos/{cat}/{cat}_{i}.jpg')
        image_cats.append(cat)

images = [Image.open(p) for p in image_paths]
image_emb = clip_model.encode(images)
print('이미지 임베딩 shape =', image_emb.shape, '  (사진 15장 × 512차원)')

**데이터부터 눈으로 봅니다.** CSV 를 만나면 `head()` 로 먼저 살펴봤듯이, 사진도 **무엇을 임베딩하는지 직접 보고** 시작합니다.

In [ ]:
# 임베딩한 사진 15장을 카테고리별로 한 줄씩 펼쳐 본다
fig, axes = plt.subplots(3, 5, figsize=(10, 6.5))
for ax, img, cat in zip(axes.ravel(), images, image_cats):
    ax.imshow(img)
    ax.set_title(cat, fontsize=10)
    ax.axis('off')
plt.suptitle('CLIP 으로 임베딩할 사진 15장')
plt.tight_layout(); plt.show()
print('해상도가 낮아 조금 흐릿하지만, CLIP 은 이 정도로도 무엇이 찍혔는지 구분한다')

In [ ]:
# 코사인 유사도로 '같은 카테고리끼리 더 비슷한지' 확인
image_sim = cosine_similarity(image_emb)
cat_arr = np.array(image_cats)
intra, inter = [], []
for a in range(len(image_paths)):
    for b in range(a + 1, len(image_paths)):
        if cat_arr[a] == cat_arr[b]:
            intra.append(image_sim[a, b])
        else:
            inter.append(image_sim[a, b])
intra_mean = np.mean(intra)
inter_mean = np.mean(inter)
print(f'같은 카테고리 평균 유사도 = {intra_mean:.3f}')
print(f'다른 카테고리 평균 유사도 = {inter_mean:.3f}')
if intra_mean > inter_mean:
    print('→ 같은 카테고리끼리가 더 비슷하다(값이 크다) — CLIP 이 사진의 뜻을 담았다')
else:
    print('→ 예상과 다르다! 카테고리별 사진이 섞였는지 image_cats 를 확인해 보자')

In [ ]:
# 512차원 이미지 임베딩을 2차원으로 줄여(UMAP) 카테고리별로 그린다
image_umap = UMAP(n_components=2, n_neighbors=5, min_dist=0.05,
                  random_state=0).fit_transform(image_emb)
color_map = {'cat': '#ef4444', 'car': '#2563eb', 'ship': '#10b981'}
label_map = {'cat': '고양이', 'car': '자동차', 'ship': '배'}

plt.figure(figsize=(7, 5.5))
for cat in categories:
    m = cat_arr == cat
    plt.scatter(image_umap[m, 0], image_umap[m, 1], color=color_map[cat],
                s=90, label=label_map[cat], alpha=0.85)
plt.title('CLIP 이미지 임베딩 (UMAP 2차원) — 카테고리별')
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2'); plt.legend(); plt.tight_layout(); plt.show()

## 해석 — 사진도 의미대로 모인다

같은 카테고리 사진끼리의 평균 유사도가 다른 카테고리보다 **뚜렷이 높습니다**. 2차원 산점도에서도 고양이·자동차·배가 **세 무리로 갈라져** 있습니다. 텍스트에서 본 것과 **똑같은 임베딩의 성질**이 이미지에서도 재현됩니다 — 그래서 이미지 검색·중복 사진 찾기·이미지 군집에 임베딩을 그대로 쓸 수 있습니다.

<img src="images/이미지_임베딩.png" width="600"/>

> 글이든 사진이든 **차원축소·시각화 도구는 UMAP 하나로 충분합니다** — 512차원이든 768차원이든 `UMAP(n_components=2, ...)` 를 그대로 적용하면 됩니다.

### 🖐️ 함께 따라하기 — 한 사진과 가장 비슷한 사진 찾기
첫 번째 사진(`image_paths[0]`, 고양이)과 **가장 비슷한 다른 사진**을 찾아보세요. `image_sim` 의 0번 행에서 자기 자신(0번)을 뺀 뒤 가장 큰 값의 인덱스를 찾고, 그 사진의 카테고리를 출력합니다. 같은 고양이 사진이 뽑히는지 확인해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) target = 0 으로 기준 사진을 정한다
# 2) sims = image_sim[target].copy() 로 0번 행을 복사하고 sims[target] = -1 로 자기 자신을 뺀다
# 3) sims.argmax() 로 가장 유사한 사진의 인덱스를 찾는다
# 4) 그 사진의 경로·카테고리·유사도를 출력해 같은 카테고리인지 확인한다

### ✅ 바로 확인 퀴즈
**1.** CLIP 임베딩에서 "비슷한 이미지"는 벡터 공간에서 어떻게 나타나나요?

<details><summary>정답 보기</summary>

**벡터가 서로 가깝게(코사인 유사도가 높게)** 나타납니다. 텍스트 임베딩과 똑같은 성질이라, 코사인 유사도·군집·UMAP 시각화를 이미지에도 그대로 적용할 수 있습니다.

</details>

**2.** 같은 카테고리 사진끼리의 평균 유사도가 다른 카테고리보다 높았습니다. 이것이 뜻하는 바는?

<details><summary>정답 보기</summary>

CLIP 이 이미지의 **의미(내용)** 를 벡터에 잘 담았다는 뜻입니다. 그래서 라벨 없이도 이미지 검색·중복 찾기·이미지 군집 같은 작업에 임베딩을 쓸 수 있습니다.

</details>

---
# 6. 텍스트-이미지 멀티모달 — 글로 사진 찾기

## CLIP 의 진짜 힘 — 공유 공간
CLIP 은 이미지와 텍스트를 **같은 벡터 공간**에 담습니다. 그래서 **글의 임베딩과 사진의 임베딩을 직접 비교**할 수 있습니다. "고양이 사진"이라는 **문장**을 임베딩하면, 실제 고양이 **사진**의 벡터와 가까워지는 것이죠. 이것이 "글로 이미지를 검색"하는 **멀티모달(multimodal)** 검색의 원리입니다.

### 문법
- **`clip_model.encode([문장, ...])`** — 사진을 넣을 때와 **똑같은 모델·똑같은 함수**에 글을 넣습니다. → `(문장 수, 512)`. 사진 임베딩과 차원이 같아야 서로 비교할 수 있습니다.
- **`cosine_similarity(글벡터, 사진벡터)`** → `(글 수, 사진 수)` 유사도 행렬.

> **CLIP 은 영어로 학습**된 모델이라, 텍스트 쿼리는 **영어**로 주는 것이 잘 맞습니다(예: `"a photo of a cat"`). 한국어로도 동작은 하지만 영어보다 정확도가 떨어집니다 — 아래 따라하기에서 **직접 확인**해 봅니다.

In [ ]:
# 영어 텍스트 쿼리를 같은 CLIP 모델로 임베딩
queries = ['a photo of a cat', 'a photo of a car', 'a photo of a ship']
query_emb = clip_model.encode(queries)

# 각 텍스트 쿼리를 15장의 이미지와 코사인 유사도로 비교 → 가장 가까운 사진
query_sim = cosine_similarity(query_emb, image_emb)
for qi, q in enumerate(queries):
    best = query_sim[qi].argmax()
    print(f'{q!r:22} -> 가장 가까운 사진: {image_cats[best]:4} (유사도 {query_sim[qi][best]:.3f})')

In [ ]:
# 각 쿼리가 카테고리별 사진과 평균적으로 얼마나 비슷한지 표로 확인
rows = []
for qi, q in enumerate(queries):
    row = {'쿼리': q}
    for cat in categories:
        row[label_map[cat]] = round(query_sim[qi][cat_arr == cat].mean(), 3)
    rows.append(row)
multimodal_table = pd.DataFrame(rows).set_index('쿼리')
print('[쿼리 × 카테고리 평균 유사도]  대각선이 가장 커야 성공')
display(multimodal_table)

## 해석 — 글이 알맞은 사진을 찾아낸다

세 영어 쿼리가 각각 **올바른 카테고리의 사진**을 가장 가깝게 찾아냈습니다. 쿼리×카테고리 평균 유사도 표에서도 **대각선(같은 뜻끼리)** 값이 가장 큽니다. 이미지와 텍스트가 **한 공간**에 있기에 가능한 일입니다. 이 원리가 "사진 검색창에 글을 입력하면 맞는 사진이 나오는" 서비스, 이미지 자동 태깅 등의 바탕이 됩니다.

### 🖐️ 함께 따라하기 — 한국어로 물으면 어떻게 될까
위 시연은 **영어** 쿼리로 검색했습니다. "CLIP 은 영어가 더 정확하다"는 말을 믿지만 말고 **같은 뜻의 한국어 쿼리**로 직접 돌려, 몇 개나 맞히는지 세어 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) ko_queries = ['고양이 사진', '자동차 사진', '배 사진'] 를 만든다
# 2) clip_model.encode(ko_queries) 로 임베딩해 ko_emb 에 담는다
# 3) cosine_similarity(ko_emb, image_emb) 로 3x15 유사도를 구한다
# 4) 쿼리마다 argmax 로 가장 가까운 사진의 카테고리를 출력하고, 영어 결과(cat/car/ship)와 비교한다

### ✅ 바로 확인 퀴즈
**1.** CLIP 으로 "글로 사진 찾기"가 가능한 근본 이유는 무엇인가요?

<details><summary>정답 보기</summary>

CLIP 이 **이미지와 텍스트를 같은(공유) 벡터 공간**에 임베딩하기 때문입니다. 그래서 문장 벡터와 사진 벡터를 **직접 코사인 유사도로 비교**할 수 있습니다.

</details>

**2.** CLIP 에 텍스트 쿼리를 줄 때 영어를 권한 이유는?

<details><summary>정답 보기</summary>

CLIP 이 주로 **영어 데이터로 학습**되어, 영어 쿼리에서 이미지와의 정렬이 더 정확하기 때문입니다. 한국어도 동작하지만 정확도가 떨어질 수 있습니다.

</details>

---
## 이번 강의 정리

| 단계 | 핵심 | 도구 |
|---|---|---|
| KMeans 군집 | 정답 없이(비지도) 비슷한 문서를 묶기; 중심점 할당·갱신 반복 | `KMeans(n_clusters, n_init)` · `fit_predict` |
| 엘보우 | k 를 몇으로? 관성이 완만해지는 팔꿈치 | `km.inertia_` · 꺾은선 |
| 실루엣 | 군집 품질(−1~1); a 작게·b 크게; 여러 k 비교 | `silhouette_score` · `silhouette_samples` |
| 대표 키워드 | 군집에 주제 이름 달기 = 토픽 모델링 기초 | `TfidfVectorizer` |
| 이미지 임베딩 | 사진도 벡터로; 같은 것끼리 가깝다 | `SentenceTransformer('clip-ViT-B-32')` · `UMAP` |
| 멀티모달 | 글과 사진을 한 공간에서 비교 → 글로 사진 검색 | CLIP · `cosine_similarity` |

이제 여러분은 임베딩을 **재는 것**(교안_01: 유사도)을 넘어 **활용**할 수 있습니다 — 라벨 없이 **묶고**(군집), 묶음에 **이름을 붙이고**(토픽 기초), **이미지와 텍스트를 넘나드는**(멀티모달) 데까지요.

> **핵심 습관**: ① 군집 수 k 는 **엘보우·실루엣·도메인 지식**을 함께 보고 정한다. ② 군집 번호엔 의미가 없으니 **대표 키워드·실제 라벨로 해석**한다. ③ 임베딩의 성질(비슷하면 가깝다)은 **텍스트·이미지에 공통**이다.

## ⏭️ 예고 — 다음 단원: 토픽 모델링 심화
오늘은 TF-IDF 로 군집에 이름을 붙이는 **기초 토픽 모델링**을 맛봤습니다. 다음 단원에서는 임베딩 기반 문서 군집화와 주제 **자동 추출·시각화**를 **본격적으로** 다뤄, 많은 문서에서 주제를 자동으로 뽑아내는 법을 배웁니다.